# Stage C 03l — Medium adaptive E25 complete-panel training

Training checkpoints are written atomically to Drive every 250 optimizer steps. If Colab disconnects, reconnect with the same Drive root and rerun all cells; the training command resumes automatically from `latest.pt` with model, optimizer, RNG, scheduler cursors, and functional stream states restored. Do not add `--no-resume`.


In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='ae72fae21ff9a0b50e4fe1d9d642c38c643b4923'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
SOURCE_ROOT='/content/drive/MyDrive/bacteria_titan_v1_ecoli_related_15gbp'
DATASET_NAME='nonoverlap_6mer_v1'
TAXONOMY_MANIFEST=f'{DRIVE_ROOT}/stage_c_dataset/manifests/accession_manifest.parquet'
ACCESSION_MANIFEST=TAXONOMY_MANIFEST
ANI_MEMBERSHIP=f'{DRIVE_ROOT}/stage_c_dataset/manifests/ani99_membership.parquet'
ANI_PAIRS=f'{DRIVE_ROOT}/inputs/ecoli_skani_triangle.tsv'
NCBI_ZIP_DIR=f'{SOURCE_ROOT}/raw/ncbi_dataset_zips'
RUN_NAME='c19_v3_medium_adaptive_e25'
RUN_ID='medium_adaptive_e25_v1'
PANEL='e25.json'


In [ ]:
from pathlib import Path
from google.colab import drive
import json, shutil, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
dataset=Path(DRIVE_ROOT)/'stage_c_dataset/ordered_streams'/DATASET_NAME
panels=Path(DRIVE_ROOT)/'study/stage_c_ecoli_medium_deep_memory_v3/panels'
PROTOCOL=repo/'studies/stage_c_ecoli_medium_deep_memory_v3/protocol.json'
RUN_SPEC=json.loads(PROTOCOL.read_text())['run_matrix'][RUN_ID]
RUN_SEED=str(RUN_SPEC['seed'])
STUDY_ROOT=Path(DRIVE_ROOT)/'study/stage_c_ecoli_medium_deep_memory_v3'
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
def run_logged(root,label,command):
    root.mkdir(parents=True,exist_ok=True)
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(root),'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        for path in (root/'FAILED.txt',root/'logs'/f'{label}.log'):
            if path.exists(): print(path.read_text(errors='replace')[-20000:])
        raise
def record_once(run_id,artifact,tier):
    marker=STUDY_ROOT/'record_markers'/f'{run_id}.json'
    if marker.exists():
        print('Ledger record already exists:',marker); return
    subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--run-id',run_id,'--evidence-tier',tier,'--artifact',str(artifact)],check=True)
    marker.parent.mkdir(parents=True,exist_ok=True)
    marker.write_text(json.dumps({'run_id':run_id,'artifact':str(artifact)},indent=2)+'\n')


In [ ]:
deep_flags=['--memory-architecture','paper_residual_mlp_v2','--memory-depth','2','--memory-expansion-factor','4','--memory-projection-convolution-kernel','4','--memory-normalize-queries-and-keys','--memory-gate-granularity','per_layer_channel','--memory-recurrence-policy','paper_exact','--memory-surprise-clip-norm','none','--memory-alpha-initial','0.001','--memory-eta-initial','0.9','--memory-theta-initial','0.001','--memory-associative-loss-reduction','sum','--memory-max-gradient-rms','none','--memory-max-gradient-rms-ratio','none','--memory-theta-max','1.0']
import torch
if 'A100' not in torch.cuda.get_device_name(0).upper(): raise RuntimeError('Select an A100 runtime.')
qualification=Path(DRIVE_ROOT)/'runs/c18_v3_medium_a100_qualification/qualification_selection.json'
q=json.loads(qualification.read_text())
if not q['passed']: raise RuntimeError('03k did not qualify this run.')
root=Path(DRIVE_ROOT)/'runs'/RUN_NAME
command=['seqtrainer-titans-stage-c-train','--dataset-dir',str(dataset),'--panel-manifest',str(panels/PANEL),'--validation-panel-manifest',str(panels/'validation.json'),'--run-dir',str(root),'--memory-mode','adaptive','--horizon','3','--batch-size',str(q['batch_size']),'--seed',RUN_SEED,'--require-panel-completion','--scheduler-policy','stateful_rotation','--scheduler-burst-segments','96','--checkpoint-every','250','--learning-rate','3e-5','--min-learning-rate','3e-6','--lr-warmup-bases','2000000','--lr-decay-bases','100000000','--weight-decay','0.1','--gradient-clip-norm','0.5','--activation',q['activation'],'--block-count','12','--d-model','256','--num-heads','8','--persistent-tokens','4',*deep_flags,'--protocol',str(PROTOCOL),'--run-id',RUN_ID]
print('Protocol-bound seed:',RUN_SEED)
print('Training resume status:', 'resuming from '+str(root/'latest.pt') if (root/'latest.pt').is_file() else 'starting a fresh run')
run_logged(root,'train_e25',command)
run_logged(root,'architecture_e25',['seqtrainer-titans-stage-c-architecture','--checkpoint',str(root/'latest.pt'),'--output-dir',str(root)])
run_logged(root,'resume_verify_e25',['seqtrainer-titans-stage-c-resume-verify','--dataset-dir',str(dataset),'--panel-manifest',str(panels/PANEL),'--checkpoint',str(root/'latest.pt'),'--output',str(root/'resume_verification.json'),'--device','cuda'])
record_once(RUN_ID,root,'exploratory')
print((root/'MODEL_ARCHITECTURE.txt').read_text()); print('SHARE THIS DIRECTORY:',root)
